In [2]:
import tensorflow as tf
from tensorflow import keras

print("=== 第 1 步：模拟一个别人训练好的模型 (Model A) ===")
# 修正 1：用专门的 Input 层代替原来的 input_shape，消灭警告！
model_A = keras.models.Sequential([
    keras.Input(shape=(28, 28)),  # <--- 新版本的标准写法
    keras.layers.Flatten(),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dense(50, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dense(10, activation="softmax") 
])

print("\n=== 第 2 步：开始你的'白嫖'手术 ===")
model_A_clone = keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

model_B = keras.models.Sequential(model_A_clone.layers[:-1])
model_B.add(keras.layers.Dense(1, activation="sigmoid"))

print("\n=== 第 3 步：见证核心魔法（冻结） ===")
for layer in model_B.layers[:-1]:
    layer.trainable = False  

# 修正 2：手动告诉 model_B 输入数据的形状，强迫它立刻把“脑袋”组装好！
model_B.build(input_shape=(None, 28, 28)) 

model_B.compile(loss="binary_crossentropy", 
                optimizer=keras.optimizers.SGD(learning_rate=0.01), 
                metrics=["accuracy"])

model_B.summary()

=== 第 1 步：模拟一个别人训练好的模型 (Model A) ===

=== 第 2 步：开始你的'白嫖'手术 ===

=== 第 3 步：见证核心魔法（冻结） ===


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,601 (326.57 KB)

 Trainable params: 51 (204.00 B)

 Non-trainable params: 83,550 (326.37 KB)

In [7]:
import tensorflow as tf
from tensorflow import keras

print("=== 1. 准备赛道：加载并清洗数据 ===")
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# 极其重要：把 0-255 的像素值缩放到 0-1 之间，神经网络最喜欢这种大小的数字
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# 切分出验证集 (Validation Set)，用来给“智能雷达”当参考
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]


print("\n=== 2. 组装超跑底盘：完美架构搭建 ===")
model = keras.models.Sequential([
    keras.Input(shape=(28, 28)), # Keras 3 标准入口
    keras.layers.Flatten(),
    
    # 第一层隐藏层：He 初始化 + 批归一化 (BN) + ELU 激活函数
    keras.layers.Dense(300, kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(), # BN 层放在 Dense 和 Activation 之间效果极佳
    keras.layers.Activation("elu"),
    
    # 第二层隐藏层：同样的满配神装
    keras.layers.Dense(100, kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("elu"),
    keras.layers.Dropout(rate=0.2),
    # 输出层：10 种衣服，所以是 10 个神经元，Softmax 输出概率
    keras.layers.Dense(10, activation="softmax")
])


print("\n=== 3. 装载 V8 引擎：优化器与编译 ===")
# 我们不用普通的 Adam，直接上带“提前刹车雷达(Nesterov)”的升级版 Nadam！
optimizer = keras.optimizers.Nadam(learning_rate=0.001)

model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])


print("\n=== 4. 配置自动驾驶仪：回调函数 (Callbacks) ===")
# 智能雷达 1：ReduceLROnPlateau (动态学习率)
# 只要验证集误差连续 4 轮没降，学习率直接砍半 (乘以 0.5)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=4, verbose=1
)

# 智能雷达 2：EarlyStopping (早停法，保命神技)
# 如果连续 10 轮都没进步，直接强行拔电源停止训练，并把最好的权重还给你！
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)


print("\n=== 5. 踩下油门，极速狂飙！ ===")
# 把数据、验证集、以及我们强大的“智能雷达”统统挂载上去
history = model.fit(
    X_train, y_train, 
    epochs=200, # 你敢设 50 轮，是因为有 EarlyStopping 保护，绝不会过拟合到死
    validation_data=(X_valid, y_valid),
    callbacks=[lr_scheduler, early_stopping]
)

print("\n=== 训练结束！可以在测试集上验收成果了 ===")
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"最终测试集准确率: {test_acc:.4f}")

=== 1. 准备赛道：加载并清洗数据 ===

=== 2. 组装超跑底盘：完美架构搭建 ===

=== 3. 装载 V8 引擎：优化器与编译 ===

=== 4. 配置自动驾驶仪：回调函数 (Callbacks) ===

=== 5. 踩下油门，极速狂飙！ ===
Epoch 1/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8197 - loss: 0.5057 - val_accuracy: 0.8478 - val_loss: 0.4193 - learning_rate: 0.0010
Epoch 2/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8511 - loss: 0.4103 - val_accuracy: 0.8722 - val_loss: 0.3581 - learning_rate: 0.0010
Epoch 3/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8634 - loss: 0.3726 - val_accuracy: 0.8596 - val_loss: 0.3805 - learning_rate: 0.0010
Epoch 4/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8744 - loss: 0.3436 - val_accuracy: 0.8744 - val_loss: 0.3553 - learning_rate: 0.0010
Epoch 5/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8804 - loss: 0.3261 - val_accuracy: 0.8706 - val_loss: 0.3468 - learning_rate: 0.0010
Epoch 6/200
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8875 - loss: 0.311

In [51]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
import os
tf.random.set_seed(42)

In [11]:
cifar10 = keras.datasets.cifar10

In [12]:
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

print(f"原始训练集形状: {X_train_full.shape}") 

X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 48s 0us/step
原始训练集形状: (50000, 32, 32, 3)


In [15]:
X_train.shape,X_test.shape,X_valid.shape,y_train.shape,y_test.shape,y_valid.shape

((45000, 32, 32, 3),
 (10000, 32, 32, 3),
 (5000, 32, 32, 3),
 (45000, 1),
 (10000, 1),
 (5000, 1))

In [21]:
y_train[0]

array([6], dtype=uint8)

In [ ]:
model = keras.models.Sequential()

model.add(keras.layers.Input(shape=[32, 32, 3]))
model.add(keras.layers.Flatten())

for i in range(20):
    model.add(keras.layers.Dense(100,kernel_initializer="he_normal"))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Activation("elu"))

model.add(keras.layers.Dense(10, activation="softmax"))

optimizer_nadam = keras.optimizers.Nadam(learning_rate=5e-5)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer_nadam,
    metrics=["accuracy"]
)

log_dir = os.path.join("logs", "fit")
tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=log_dir, 
    histogram_freq=0  # 每一轮都统计一下权重的直方图（可选，想看详细的就设为1）
)

callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True),
          tensorboard_cb]

history = model.fit(X_train,y_train,epochs=100,
                    validation_data=(X_valid,y_valid),
                   callbacks=callbacks)

In [46]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 16099), started 0:14:30 ago. (Use '!kill 16099' to kill it.)

In [53]:
model.evaluate(X_valid, y_valid)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 930us/step - accuracy: 0.4982 - loss: 1.4238


[1.4237569570541382, 0.498199999332428]

In [57]:
model = keras.models.Sequential()

model.add(keras.layers.Input(shape=[32, 32, 3]))
model.add(keras.layers.Flatten())

for i in range(20):
    model.add(keras.layers.Dense(100,kernel_initializer="lecun_normal",activation = "selu"))

model.add(keras.layers.Dense(10, activation="softmax"))

optimizer_nadam = keras.optimizers.Nadam(learning_rate=5e-5)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer_nadam,
    metrics=["accuracy"]
)

log_dir = os.path.join("logs", "fit")
tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=log_dir, 
    histogram_freq=0  # 每一轮都统计一下权重的直方图（可选，想看详细的就设为1）
)

callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True),
          tensorboard_cb]

history = model.fit(X_train,y_train,epochs=100,
                    validation_data=(X_valid,y_valid),
                   callbacks=callbacks)

Epoch 1/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.3240 - loss: 1.8627 - val_accuracy: 0.3758 - val_loss: 1.7063
Epoch 2/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3891 - loss: 1.6937 - val_accuracy: 0.4018 - val_loss: 1.6462
Epoch 3/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4150 - loss: 1.6230 - val_accuracy: 0.4108 - val_loss: 1.6265
Epoch 4/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4348 - loss: 1.5702 - val_accuracy: 0.4258 - val_loss: 1.5848
Epoch 5/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4501 - loss: 1.5320 - val_accuracy: 0.4394 - val_loss: 1.5624
Epoch 6/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4616 - loss: 1.4985 - val_accuracy: 0.4500 - val_loss: 1.5422
Epoch 7/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4742 - loss: 1.4701 - val_accuracy: 0.4472 - val_loss: 1.5357
Epoch 8/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4838 - loss: 1

In [58]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 16099), started 0:36:22 ago. (Use '!kill 16099' to kill it.)

In [59]:
model.evaluate(X_valid, y_valid)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4826 - loss: 1.4747


[1.474725604057312, 0.48260000348091125]

In [70]:
model = keras.models.Sequential()

model.add(keras.layers.Input(shape=[32, 32, 3]))
model.add(keras.layers.Flatten())

for i in range(20):
    model.add(keras.layers.Dense(100,kernel_initializer="he_normal",activation = "selu"))
    model.add(keras.layers.AlphaDropout(rate=0.1))
model.add(keras.layers.Dense(10, activation="softmax"))

optimizer_nadam = keras.optimizers.Nadam(learning_rate=5e-5)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer_nadam,
    metrics=["accuracy"]
)

log_dir = os.path.join("logs", "fit")
tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=log_dir, 
    histogram_freq=0  # 每一轮都统计一下权重的直方图（可选，想看详细的就设为1）
)

callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True),
          tensorboard_cb]

history = model.fit(X_train,y_train,epochs=100,
                    validation_data=(X_valid,y_valid),
                   callbacks=callbacks)

Epoch 1/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.0986 - loss: 3.8117 - val_accuracy: 0.1052 - val_loss: 3.4574
Epoch 2/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.1014 - loss: 2.7603 - val_accuracy: 0.0892 - val_loss: 2.9739
Epoch 3/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0984 - loss: 2.4865 - val_accuracy: 0.1218 - val_loss: 2.8034
Epoch 4/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0995 - loss: 2.3875 - val_accuracy: 0.1004 - val_loss: 2.6093
Epoch 5/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.1020 - loss: 2.3433 - val_accuracy: 0.1008 - val_loss: 2.4408
Epoch 6/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.1002 - loss: 2.3264 - val_accuracy: 0.1074 - val_loss: 2.3130
Epoch 7/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0997 - loss: 2.3157 - val_accuracy: 0.1010 - val_loss: 2.3460
Epoch 8/100
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.1014 - loss: 2

In [71]:
model.evaluate(X_valid, y_valid)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 755us/step - accuracy: 0.1074 - loss: 2.3130


[2.3130180835723877, 0.10740000009536743]

In [72]:
import numpy as np

print("=== 1. 先看看普通模式的成绩（作为对照组） ===")
# 普通预测：Keras 会自动关掉你的 AlphaDropout
loss, acc = model.evaluate(X_test, y_test)
print(f"普通模式测试集准确率: {acc:.4f}\n")


print("=== 2. 🎲 开启 MC Dropout 魔法（不重新训练！） ===")
# 核心黑科技：用 model(..., training=True) 强行让网络以为还在训练，
# 从而逼迫 AlphaDropout 继续工作！我们让它连续跑 100 次。
mc_predictions = np.stack([model(X_test, training=True) for _ in range(100)])

# mc_predictions 现在的形状是 (100次, 10000张图片, 10个类别概率)

# 民主投票：把这 100 次“残缺模型”的预测概率求平均
mc_ensemble_preds = mc_predictions.mean(axis=0)

# 选出平均概率最大的那个类别作为最终答案
final_preds = np.argmax(mc_ensemble_preds, axis=1)

# 计算最终准确率
mc_accuracy = np.mean(final_preds == y_test.flatten())

print(f"🚀 MC Dropout 加持后的最终准确率: {mc_accuracy:.4f}")

=== 1. 先看看普通模式的成绩（作为对照组） ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step - accuracy: 0.1067 - loss: 2.3185
普通模式测试集准确率: 0.1067

=== 2. 🎲 开启 MC Dropout 魔法（不重新训练！） ===
🚀 MC Dropout 加持后的最终准确率: 0.0997


In [73]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.Sequential([
    # 🌟 第一层：不再是 Flatten！直接用 64 个 3x3 的放大镜去扫描 32x32 的彩色图片！
    # 填充 padding="same" 意味着扫描后图片大小不变。
    keras.layers.Conv2D(64, kernel_size=3, activation="relu", padding="same", input_shape=[32, 32, 3]),
    
    # 🗜️ 压缩机：长宽各砍一半（变成 16x16）
    keras.layers.MaxPooling2D(pool_size=2),
    
    # 🌟 第二层：用 128 个放大镜继续扫描这缩小后的 16x16 图片
    keras.layers.Conv2D(128, kernel_size=3, activation="relu", padding="same"),
    
    # 🗜️ 压缩机：长宽再砍一半（变成 8x8）
    keras.layers.MaxPooling2D(pool_size=2),
    
    # 🧱 最后收尾：经过了前面的浓缩提炼，我们现在可以放心地把它拍扁了
    keras.layers.Flatten(),
    
    # 用普通的 Dense 层做最后的 10 分类输出
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [75]:
import tensorflow as tf
from tensorflow import keras

print("=== 1. 召唤 CIFAR-10 数据集 ===")
cifar10 = keras.datasets.cifar10
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

# 极其关键：将 0-255 的像素值压缩到 0-1 之间
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# 割肉：切分出 5000 张作为验证集
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]


print("\n=== 2. 搭建 CNN 核心架构 ===")
model = keras.models.Sequential([
    # 🌟 第一层：64 个 3x3 的放大镜，保留边缘填充 (same)
    keras.layers.Conv2D(64, kernel_size=3, activation="relu", padding="same", input_shape=[32, 32, 3]),
    # 🗜️ 压缩机：长宽砍半 (32x32 变成 16x16)
    keras.layers.MaxPooling2D(pool_size=2),
    
    # 🌟 第二层：128 个放大镜继续提炼高级特征
    keras.layers.Conv2D(128, kernel_size=3, activation="relu", padding="same"),
    # 🗜️ 压缩机：长宽再砍半 (16x16 变成 8x8)
    keras.layers.MaxPooling2D(pool_size=2),
    
    # 🧱 收尾：此时图片已经被浓缩成了 8x8 的极品特征图，可以放心拍扁了
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

# 打印一下大楼结构，你会发现参数量比之前少，但结构极其精悍！
model.summary()


print("\n=== 3. 挂载引擎与雷达 ===")
# 因为 CNN 极其稳定，我们甚至不需要搞几万分之一的微小学习率，直接用默认的 Nadam 就行！
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam",
              metrics=["accuracy"])

# 保命雷达 (耐心值设为 5，拿到最好权重)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)


print("\n=== 4. 🚀 点火起飞！体验降维打击 ===")
# 先设定跑 15 轮，你盯着每一轮的准确率看！
history = model.fit(X_train, y_train, epochs=15,
                    validation_data=(X_valid, y_valid),
                    callbacks=[early_stopping_cb])


print("\n=== 5. 终极验收 ===")
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"🎉 CNN 首次出战，CIFAR-10 测试集最终准确率: {test_acc:.4f}")

=== 1. 召唤 CIFAR-10 数据集 ===

=== 2. 搭建 CNN 核心架构 ===


Model: "sequential_30"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_22 (Flatten)            │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_345 (Dense)               │ (None, 64)             │       524,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_346 (Dense)               │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600,650 (2.29 MB)

 Trainable params: 600,650 (2.29 MB)

 Non-trainable params: 0 (0.00 B)


=== 3. 挂载引擎与雷达 ===

=== 4. 🚀 点火起飞！体验降维打击 ===
Epoch 1/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 31s 22ms/step - accuracy: 0.5298 - loss: 1.3194 - val_accuracy: 0.6458 - val_loss: 1.0087
Epoch 2/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 34s 24ms/step - accuracy: 0.6654 - loss: 0.9545 - val_accuracy: 0.6736 - val_loss: 0.9445
Epoch 3/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 37s 26ms/step - accuracy: 0.7179 - loss: 0.8084 - val_accuracy: 0.6882 - val_loss: 0.9279
Epoch 4/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 37s 26ms/step - accuracy: 0.7612 - loss: 0.6941 - val_accuracy: 0.6996 - val_loss: 0.9015
Epoch 5/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 37s 26ms/step - accuracy: 0.7975 - loss: 0.5920 - val_accuracy: 0.7116 - val_loss: 0.9059
Epoch 6/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 38s 27ms/step - accuracy: 0.8287 - loss: 0.4996 - val_accuracy: 0.7048 - val_loss: 0.9912
Epoch 7/15
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 38s 27ms/step - accuracy: 0.8579 - loss: 0.4207 - val_accuracy: 0.6906 - val_loss: 1.1119
Epoch 8/15
1407/1407 ━━━━━━━━━